# Random Forest Regressor for Petrol Consumption Prediction

This notebook demonstrates the use of Random Forest Regressor to predict petrol consumption based on various features. The dataset contains information about petrol consumption in different countries.

## Table of Contents
- [1. Import Libraries](#1.-Import-Libraries)
- [2. Load and Explore Data](#2.-Load-and-Explore-Data)
- [3. Data Preprocessing](#3.-Data-Preprocessing)
- [4. Train-Test Split](#4.-Train-Test-Split)
- [5. Feature Scaling](#5.-Feature-Scaling)
- [6. Model Training](#6.-Model-Training)
- [7. Model Evaluation](#7.-Model-Evaluation)
- [8. Hyperparameter Tuning](#8.-Hyperparameter-Tuning)
- [9. Feature Importance Analysis](#9.-Feature-Importance-Analysis)
- [10. Predictions and Visualization](#10.-Predictions-and-Visualization)

In [1]:
# 1. Import Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                r2_score, explained_variance_score)

# Set style for plots
sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")

In [2]:
# 2. Load and Explore Data

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load the dataset from Kaggle
file_path = "petrol_consumption.csv"
datasets = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "harinir/petrol-consumption",
    file_path
)

# Display basic information about the dataset
print("Dataset shape:", datasets.shape)
print("\nFirst 5 rows:")
datasets.head()

In [3]:
# 2.1 Data Exploration

print("\nDataset Info:")
datasets.info()

print("\n\nDescriptive Statistics:")
datasets.describe()

print("\n\nMissing Values:")
datasets.isnull().sum()

In [4]:
# 2.2 Data Visualization

plt.figure(figsize=(15, 10))

# Plot distributions of all features
for i, col in enumerate(datasets.columns):
    plt.subplot(2, 3, i+1)
    sns.histplot(datasets[col], kde=True)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(datasets.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

In [5]:
# 3. Data Preprocessing

# Separate features and target variable
X = datasets.iloc[:, 0:4].values
y = datasets.iloc[:, 4].values

# Feature names for reference
feature_names = datasets.columns[:4]
target_name = datasets.columns[4]

print(f"Features: {feature_names}")
print(f"Target: {target_name}")

In [6]:
# 4. Train-Test Split

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

In [7]:
# 5. Feature Scaling

# Standardize features by removing the mean and scaling to unit variance
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)  # Important: use transform, not fit_transform

print("Feature scaling applied to both training and test sets")

In [8]:
# 6. Model Training

# Initialize Random Forest Regressor with default parameters
regressor = RandomForestRegressor(
    n_estimators=100,  # Number of trees in the forest
    random_state=42,   # For reproducibility
    n_jobs=-1         # Use all available cores
)

# Train the model
regressor.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = regressor.predict(X_test_scaled)

print("Model trained successfully!")

In [9]:
# 7. Model Evaluation

def evaluate_model(y_true, y_pred, set_name):
    """Evaluate model performance with multiple metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)

    print(f"\n{set_name} Set Performance:")
    print(f"  MAE: {mae:.4f}")
    print(f"  MSE: {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R² Score: {r2:.4f}")
    print(f"  Explained Variance: {evs:.4f}")

    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'EVS': evs
    }

# Evaluate on training set
train_pred = regressor.predict(X_train_scaled)
train_metrics = evaluate_model(y_train, train_pred, "Training")

# Evaluate on test set
test_metrics = evaluate_model(y_test, y_pred, "Test")

In [10]:
# 8. Hyperparameter Tuning using GridSearchCV

print("\nPerforming hyperparameter tuning...")

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    n_jobs=-1,
    verbose=2,
    scoring='neg_mean_squared_error'
)

# Perform grid search
grid_search.fit(X_train_scaled, y_train)

# Best parameters and score
print("\nBest parameters found:")
print(grid_search.best_params_)

print(f"\nBest negative MSE score: {grid_search.best_score_:.4f}")
print(f"Best RMSE score: {np.sqrt(-grid_search.best_score_):.4f}")

# Train model with best parameters
best_regressor = grid_search.best_estimator_
y_pred_tuned = best_regressor.predict(X_test_scaled)

# Evaluate tuned model
print("\nTuned Model Performance:")
evaluate_model(y_test, y_pred_tuned, "Test (Tuned)")

In [11]:
# 9. Feature Importance Analysis

print("\nFeature Importance Analysis:")

# Get feature importances from the best model
importances = best_regressor.feature_importances_
indices = np.argsort(importances)[::-1]

# Print feature ranking
print("\nFeature ranking:")
for f in range(X.shape[1]):
    print(f"{f + 1}. {feature_names[indices[f]]} ({importances[indices[f]]:.4f})")

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), [feature_names[i] for i in indices], rotation=45)
plt.xlim([-1, X.shape[1]])
plt.ylabel("Importance Score")
plt.show()

In [12]:
# 10. Predictions and Visualization

# Create a DataFrame for actual vs predicted values
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred_tuned
})

# Plot actual vs predicted values
plt.figure(figsize=(10, 6))
sns.scatterplot(data=results, x='Actual', y='Predicted')
plt.plot([results['Actual'].min(), results['Actual'].max()],
         [results['Actual'].min(), results['Actual'].max()], 'r--', lw=2)
plt.xlabel('Actual Petrol Consumption')
plt.ylabel('Predicted Petrol Consumption')
plt.title('Actual vs Predicted Petrol Consumption')
plt.show()

# Plot residuals
results['Residuals'] = results['Actual'] - results['Predicted']

plt.figure(figsize=(10, 6))
sns.histplot(results['Residuals'], kde=True)
plt.xlabel('Residuals')
plt.title('Distribution of Residuals (Errors)')
plt.show()

In [ ]:
# Summary and Next Steps

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print("\nThis notebook demonstrated:")
print("1. Loading and exploring the petrol consumption dataset")
print("2. Data preprocessing and feature scaling")
print("3. Training a Random Forest Regressor model")
print("4. Evaluating model performance with multiple metrics")
print("5. Hyperparameter tuning using GridSearchCV")
print("6. Feature importance analysis")
print("7. Visualizing predictions and residuals")

print("\nNext Steps:")
print("- Try different preprocessing techniques (e.g., normalization)")
print("- Experiment with other ensemble methods (e.g., Gradient Boosting)")
print("- Perform more extensive feature engineering")
print("- Use cross-validation for more robust evaluation")